In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from neural_tangents import stax
import jax.numpy as jnp

# Load dataset
data = pd.read_csv('AIDS_Classification.csv')

# Define features and target
X = data.drop(columns=['infected'])
y = data['infected']

# Normalize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Define NTK model
init_fn, apply_fn, kernel_fn = stax.serial(
    stax.Dense(128), stax.Relu(),
    stax.Dense(64), stax.Relu(),
    stax.Dense(1)
)

# Compute the NTK kernel matrix
kernel_train = kernel_fn(X_train, X_train, "ntk")
kernel_test = kernel_fn(X_test, X_train, "ntk")

# Solve for the kernel regression coefficients
coefficients = np.linalg.solve(kernel_train + 1e-3 * np.eye(len(kernel_train)), y_train)

# Make predictions on the test set
predictions = jnp.dot(kernel_test, coefficients)
predictions = jnp.round(predictions)  # Convert predictions to binary

# Evaluate the model
accuracy = accuracy_score(y_test, predictions)
print(f"Accuracy: {accuracy:.2f}")



Accuracy: 0.86


In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, f1_score
from neural_tangents import stax
import jax.numpy as jnp
import joblib

# Load dataset
data = pd.read_csv('AIDS_Classification.csv')

# Define features and target
X = data.drop(columns=['infected'])
y = data['infected']

# Normalize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Set up K-fold cross-validation
kf = KFold(n_splits=10, shuffle=True, random_state=42)  # Change n_splits to 15 for 15-fold cross-validation

accuracies = []
precisions = []
f1_scores = []

# Define NTK model
init_fn, apply_fn, kernel_fn = stax.serial(
    stax.Dense(128), stax.Relu(),
    stax.Dense(64), stax.Relu(),
    stax.Dense(1)
)

best_fold = None
best_accuracy = 0
best_coefficients = None
best_kernel_train = None

# K-fold cross-validation
for fold, (train_index, test_index) in enumerate(kf.split(X_scaled), 1):
    X_train, X_test = X_scaled[train_index], X_scaled[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    # Compute the NTK kernel matrix
    kernel_train = kernel_fn(X_train, X_train, "ntk")
    kernel_test = kernel_fn(X_test, X_train, "ntk")

    # Solve for the kernel regression coefficients
    coefficients = np.linalg.solve(kernel_train + 1e-3 * np.eye(len(kernel_train)), y_train)

    # Make predictions on the test set
    predictions = jnp.dot(kernel_test, coefficients)
    predictions = jnp.round(predictions)  # Convert predictions to binary

    # Evaluate the model on this fold
    accuracy = accuracy_score(y_test, predictions)

    # For multiclass classification, use 'macro' for precision and F1 score
    precision = precision_score(y_test, predictions, average='macro')
    f1 = f1_score(y_test, predictions, average='macro')

    # Store the metrics for this fold
    accuracies.append(accuracy)
    precisions.append(precision)
    f1_scores.append(f1)

    # Save the best model based on accuracy
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_fold = fold
        best_coefficients = coefficients
        best_kernel_train = kernel_train

    # Print the metrics for this fold
    print(f"Fold {fold}:")
    print(f"  Accuracy: {accuracy:.2f}")
    print(f"  Precision (macro): {precision:.2f}")
    print(f"  F1 Score (macro): {f1:.2f}")
    print("-" * 40)

# Save the best model to a .pkl file
model_data = {
    'scaler': scaler,
    'coefficients': best_coefficients,
    'kernel_train': best_kernel_train
}
joblib.dump(model_data, 'best_ntk_model.pkl')
print(f"Best model saved from Fold {best_fold} with accuracy {best_accuracy:.2f}.")

# Calculate and print the average metrics across all folds
average_accuracy = np.mean(accuracies)
average_precision = np.mean(precisions)
average_f1_score = np.mean(f1_scores)

print("\nAverage Metrics across all folds:")
print(f"Average Accuracy: {average_accuracy:.2f}")
print(f"Average Precision (macro): {average_precision:.2f}")
print(f"Average F1 Score (macro): {average_f1_score:.2f}")


Fold 1:
  Accuracy: 0.84
  Precision (macro): 0.83
  F1 Score (macro): 0.75
----------------------------------------
Fold 2:
  Accuracy: 0.88
  Precision (macro): 0.84
  F1 Score (macro): 0.83
----------------------------------------
Fold 3:
  Accuracy: 0.88
  Precision (macro): 0.58
  F1 Score (macro): 0.55
----------------------------------------
Fold 4:
  Accuracy: 0.89
  Precision (macro): 0.86
  F1 Score (macro): 0.84
----------------------------------------
Fold 5:
  Accuracy: 0.90
  Precision (macro): 0.86
  F1 Score (macro): 0.83
----------------------------------------
Fold 6:
  Accuracy: 0.86
  Precision (macro): 0.83
  F1 Score (macro): 0.82
----------------------------------------
Fold 7:
  Accuracy: 0.86
  Precision (macro): 0.82
  F1 Score (macro): 0.80
----------------------------------------
Fold 8:
  Accuracy: 0.88
  Precision (macro): 0.83
  F1 Score (macro): 0.83
----------------------------------------
Fold 9:
  Accuracy: 0.85
  Precision (macro): 0.82
  F1 Score (

In [2]:
X.shape

(2139, 22)